In [1]:
import pandas as pd
import numpy as np 
import math
import yaml
from sklearn.preprocessing import OneHotEncoder
from copy import deepcopy
from typing import Tuple, Any, Dict, List
from src.utils import load_config, load_data, serialize_data, deserialize_data, get_project_root


In [2]:
root = get_project_root()

In [3]:
def save_to_config(key: str, value: any, filename: str = "config.yaml"):
    """
    Menyimpan key dan value baru LANGSUNG ke file config.yaml
    tanpa merusak path yang sudah ada.
    """
    path_config = get_project_root() / "config" / filename
    
    with open(path_config, "r") as file:
        raw_config = yaml.safe_load(file) or {}
        
    raw_config[key] = value
    
    with open(path_config, "w") as file:
        yaml.dump(raw_config, file, default_flow_style=False, sort_keys=True)
        
    print(f"Berhasil menyimpan permanen: '{key}' ke {filename}")

In [4]:
config = load_config()

In [5]:
config

{'columns_cat': ['person_home_ownership',
  'loan_intent',
  'loan_grade',
  'cb_person_default_on_file'],
 'columns_num': ['person_age',
  'person_income',
  'person_emp_length',
  'loan_amnt',
  'loan_int_rate',
  'loan_percent_income',
  'cb_person_cred_hist_length'],
 'columns_predictors': ['person_age',
  'person_income',
  'person_emp_length',
  'loan_amnt',
  'loan_int_rate',
  'loan_percent_income',
  'cb_person_cred_hist_length',
  'person_home_ownership',
  'loan_intent',
  'loan_grade',
  'cb_person_default_on_file'],
 'path_imputer_median': 'models/imputer_median.pkl',
 'path_imputer_mode': 'models/imputer_mode.pkl',
 'path_raw_data': 'data/raw/credit_risk_dataset.csv',
 'path_test_clean': ['data/processed/X_test_clean.pkl',
  'data/processed/y_test_clean.pkl'],
 'path_test_set': ['data/interim/X_test.pkl', 'data/interim/y_test.pkl'],
 'path_train_clean': ['data/processed/X_train_clean.pkl',
  'data/processed/y_train_clean.pkl'],
 'path_train_set': ['data/interim/X_train.pk

In [6]:
path_X_train, path_y_train = config["path_train_set"]
path_X_valid, path_y_valid = config["path_valid_set"]
path_X_test, path_y_test = config["path_test_set"]

In [7]:
X_train = deserialize_data(path_X_train)
y_train = deserialize_data(path_y_train)

X_valid = deserialize_data(path_X_valid)
y_valid = deserialize_data(path_y_valid)

X_test = deserialize_data(path_X_test)
y_test = deserialize_data(path_y_test)

In [8]:
X_train.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
29762,45,37500,MORTGAGE,1.0,DEBTCONSOLIDATION,B,5000,11.49,0.13,N,16
2714,25,50000,RENT,5.0,PERSONAL,A,12000,7.88,0.24,N,2
50,24,78000,RENT,4.0,DEBTCONSOLIDATION,D,30000,NaN,0.38,Y,4
28458,31,78504,RENT,2.0,EDUCATION,C,10000,11.41,0.13,N,7
3674,26,14000,RENT,2.0,VENTURE,B,4000,NaN,0.29,N,3


In [9]:
y_train.head()

29762    0
2714     0
50       1
28458    0
3674     1
Name: loan_status, dtype: int64

In [10]:
def drop_duplicate_data(X: pd.DataFrame, y: pd.Series) -> Tuple[pd.DataFrame, pd.Series]:

    if not isinstance(X, pd.DataFrame):
        raise TypeError("Fungsi drop_duplicate_data: Parameter (X) harus bertipe pd.DataFrame.")
    elif not isinstance(y, pd.Series):
        raise TypeError("Fungsi drop_duplicate_data: Parameter (y) harus bertipe pd.Series")
    else:
        print("Fungsi drop_duplicate_data: parameter telah divalidasi.")
    
    X = X.copy()
    y = y.copy()

    print(f"Fungsi drop_duplicate_data: shape dataset sebelum dropping duplicate adalah ->        {X.shape}.")

    X_duplicate = X[X.duplicated()]
    print(f"Fungsi drop_duplicate_data: shape dari data yang duplicate adalah ->                  {X_duplicate.shape}.")

    X_clean = (X.shape[0] - X_duplicate.shape[0], X.shape[1])
    print(f"Fungsi drop_duplicate_data: shape dataset setelah drop duplicate seharusnya adalah -> {X_clean}.")

    X.drop_duplicates(inplace=True)
    y = y.loc[X.index]

    print(f"Fungsi drop_duplicate_data: shape dataset setelah dropping duplicate adalah ->        {X.shape}.")

    return X, y


In [11]:
X_train, y_train = drop_duplicate_data(X_train, y_train)

Fungsi drop_duplicate_data: parameter telah divalidasi.
Fungsi drop_duplicate_data: shape dataset sebelum dropping duplicate adalah ->        (26064, 11).
Fungsi drop_duplicate_data: shape dari data yang duplicate adalah ->                  (110, 11).
Fungsi drop_duplicate_data: shape dataset setelah drop duplicate seharusnya adalah -> (25954, 11).
Fungsi drop_duplicate_data: shape dataset setelah dropping duplicate adalah ->        (25954, 11).


In [12]:
X_valid, y_valid = drop_duplicate_data(X_valid, y_valid)

Fungsi drop_duplicate_data: parameter telah divalidasi.
Fungsi drop_duplicate_data: shape dataset sebelum dropping duplicate adalah ->        (3258, 11).
Fungsi drop_duplicate_data: shape dari data yang duplicate adalah ->                  (2, 11).
Fungsi drop_duplicate_data: shape dataset setelah drop duplicate seharusnya adalah -> (3256, 11).
Fungsi drop_duplicate_data: shape dataset setelah dropping duplicate adalah ->        (3256, 11).


In [13]:
X_test, y_test = drop_duplicate_data(X_test, y_test)

Fungsi drop_duplicate_data: parameter telah divalidasi.
Fungsi drop_duplicate_data: shape dataset sebelum dropping duplicate adalah ->        (3259, 11).
Fungsi drop_duplicate_data: shape dari data yang duplicate adalah ->                  (0, 11).
Fungsi drop_duplicate_data: shape dataset setelah drop duplicate seharusnya adalah -> (3259, 11).
Fungsi drop_duplicate_data: shape dataset setelah dropping duplicate adalah ->        (3259, 11).


In [14]:
def filter_domain_outliers(data: pd.DataFrame) -> pd.DataFrame:

    if not isinstance(data, pd.DataFrame):
        raise TypeError("Fungsi filter_domain_outliers: parameter data harus bertipe DataFrame.")
    
    print("Fungsi filter_domain_outliers: memulai pengecekan logika umur")
    data = data.copy()

    age_outliers = data['person_age'] > 100
    print(f"   -> Ditemukan {age_outliers.sum()} data dengan person_age > 100. Mengubah ke NaN.")
    data.loc[age_outliers, 'person_age'] = np.nan

    emp_length_outliers = (data['person_emp_length'] > 60) | (data['person_emp_length'] >= data['person_age'])
    print(f"   -> Ditemukan {emp_length_outliers.sum()} data dengan person_emp_length yang mustahil. Mengubah ke NaN.")
    data.loc[emp_length_outliers, 'person_emp_length'] = np.nan

    print("Fungsi filter_domain_outliers: pembersihan selesai.\n")
    return data

In [15]:
X_train = filter_domain_outliers(X_train)

Fungsi filter_domain_outliers: memulai pengecekan logika umur
   -> Ditemukan 3 data dengan person_age > 100. Mengubah ke NaN.
   -> Ditemukan 1 data dengan person_emp_length yang mustahil. Mengubah ke NaN.
Fungsi filter_domain_outliers: pembersihan selesai.



In [16]:
X_valid = filter_domain_outliers(X_valid)

Fungsi filter_domain_outliers: memulai pengecekan logika umur
   -> Ditemukan 0 data dengan person_age > 100. Mengubah ke NaN.
   -> Ditemukan 1 data dengan person_emp_length yang mustahil. Mengubah ke NaN.
Fungsi filter_domain_outliers: pembersihan selesai.



In [17]:
X_test = filter_domain_outliers(X_test)

Fungsi filter_domain_outliers: memulai pengecekan logika umur
   -> Ditemukan 2 data dengan person_age > 100. Mengubah ke NaN.
   -> Ditemukan 0 data dengan person_emp_length yang mustahil. Mengubah ke NaN.
Fungsi filter_domain_outliers: pembersihan selesai.



In [18]:
def fit_median_imputation(data: pd.DataFrame, subset_data: List[str]) -> Dict[str, float]:
    
    if not isinstance(data, pd.DataFrame):
        raise TypeError("Fungsi fit_median_imputation: parameter data harus bertipe DataFrame.")
    if not isinstance(subset_data, list):
        raise TypeError("Fungsi fit_median_imputation: parameter subset_data harus bertipe list.")
    
    print("Fungsi fit_median_imputation: parameter telah divalidasi.")

    imputation_data = {}

    if "person_emp_length" in subset_data and "person_age" in data.columns:

        valid_data = data.dropna(subset=["person_age", "person_emp_length"])

        starting_age = valid_data["person_age"] - valid_data["person_emp_length"]

        median_starting_age = starting_age.median()

        imputation_data['median_starting_age'] = median_starting_age
        print(f"   -> [Domain Rule] Median usia mulai bekerja didapatkan: {median_starting_age} tahun.")
    
    for col in subset_data:
        if col != "person_emp_length":
            imputation_data[col] = data[col].median()
    
    print(f"Fungsi fit_median_imputation: proses fitting telah selesai, berikut hasilnya {imputation_data}.")
    return imputation_data

In [19]:
def transform_median_imputation(data: pd.DataFrame, imputation_data: Dict[str, float]) -> pd.DataFrame:

    if not isinstance(data, pd.DataFrame):
        raise TypeError("Fungsi transform_median_imputation: parameter data harus bertipe DataFrame.")
    if not isinstance(imputation_data, dict):
        raise TypeError("Fungsi transform_median_imputation: parameter imputation_data harus bertipe dict.")
    
    print("Fungsi transform_median_imputation: parameter telah divalidasi.")
    data = data.copy()

    cols_to_check = [col for col in imputation_data.keys() if col != "median_starting_age"]
    if "median_starting_age" in imputation_data:
        cols_to_check.append("person_emp_length")
    
    print("Fungsi transform_median_imputation: informasi count na sebelum dilakukan imputasi:")
    print(data[cols_to_check].isna().sum())
    print("")

    standard_impute_dict = {k:v for k, v in imputation_data.items() if k != "median_starting_age"}
    if standard_impute_dict:
        data.fillna(standard_impute_dict, inplace=True)

    if "median_starting_age" in imputation_data and "person_emp_length" in data.columns:

        missing_emp_idx = data["person_emp_length"].isna()

        if missing_emp_idx.sum() > 0:

            calculated_emp = data.loc[missing_emp_idx, "person_age"] - imputation_data["median_starting_age"]

            calculated_emp = calculated_emp.apply(lambda x: max(0, x))

            data.loc[missing_emp_idx, "person_emp_length"] = calculated_emp
        
    
    
    print("Fungsi transform_median_imputation: informasi count na setelah dilakukan imputasi:")
    print(data[cols_to_check].isna().sum())
    print("") 
    
    return data

In [20]:
cat_col = [
    "person_home_ownership",
    "loan_intent",
    "loan_grade",
    "cb_person_default_on_file"
]

num_col = [
    "person_age",
    "person_income",
    "person_emp_length",
    "loan_amnt",
    "loan_int_rate",
    "loan_percent_income",
    "cb_person_cred_hist_length"
]

predictors = num_col + cat_col

save_to_config("columns_predictors", predictors)
save_to_config("columns_num", num_col)  
save_to_config("columns_cat", cat_col)


Berhasil menyimpan permanen: 'columns_predictors' ke config.yaml
Berhasil menyimpan permanen: 'columns_num' ke config.yaml
Berhasil menyimpan permanen: 'columns_cat' ke config.yaml


In [21]:
config = load_config()

In [22]:
config

{'columns_cat': ['person_home_ownership',
  'loan_intent',
  'loan_grade',
  'cb_person_default_on_file'],
 'columns_num': ['person_age',
  'person_income',
  'person_emp_length',
  'loan_amnt',
  'loan_int_rate',
  'loan_percent_income',
  'cb_person_cred_hist_length'],
 'columns_predictors': ['person_age',
  'person_income',
  'person_emp_length',
  'loan_amnt',
  'loan_int_rate',
  'loan_percent_income',
  'cb_person_cred_hist_length',
  'person_home_ownership',
  'loan_intent',
  'loan_grade',
  'cb_person_default_on_file'],
 'path_imputer_median': 'models/imputer_median.pkl',
 'path_imputer_mode': 'models/imputer_mode.pkl',
 'path_raw_data': 'data/raw/credit_risk_dataset.csv',
 'path_test_clean': ['data/processed/X_test_clean.pkl',
  'data/processed/y_test_clean.pkl'],
 'path_test_set': ['data/interim/X_test.pkl', 'data/interim/y_test.pkl'],
 'path_train_clean': ['data/processed/X_train_clean.pkl',
  'data/processed/y_train_clean.pkl'],
 'path_train_set': ['data/interim/X_train.pk

In [23]:
PATH_IMPUTER_MEDIAN = "models/imputer_median.pkl"

In [24]:
save_to_config(
    key="path_imputer_median",
    value=PATH_IMPUTER_MEDIAN
)

Berhasil menyimpan permanen: 'path_imputer_median' ke config.yaml


In [25]:
config = load_config()

In [26]:
config

{'columns_cat': ['person_home_ownership',
  'loan_intent',
  'loan_grade',
  'cb_person_default_on_file'],
 'columns_num': ['person_age',
  'person_income',
  'person_emp_length',
  'loan_amnt',
  'loan_int_rate',
  'loan_percent_income',
  'cb_person_cred_hist_length'],
 'columns_predictors': ['person_age',
  'person_income',
  'person_emp_length',
  'loan_amnt',
  'loan_int_rate',
  'loan_percent_income',
  'cb_person_cred_hist_length',
  'person_home_ownership',
  'loan_intent',
  'loan_grade',
  'cb_person_default_on_file'],
 'path_imputer_median': 'models/imputer_median.pkl',
 'path_imputer_mode': 'models/imputer_mode.pkl',
 'path_raw_data': 'data/raw/credit_risk_dataset.csv',
 'path_test_clean': ['data/processed/X_test_clean.pkl',
  'data/processed/y_test_clean.pkl'],
 'path_test_set': ['data/interim/X_test.pkl', 'data/interim/y_test.pkl'],
 'path_train_clean': ['data/processed/X_train_clean.pkl',
  'data/processed/y_train_clean.pkl'],
 'path_train_set': ['data/interim/X_train.pk

In [27]:
imputer_median_dict = fit_median_imputation(X_train, config["columns_num"])

Fungsi fit_median_imputation: parameter telah divalidasi.
   -> [Domain Rule] Median usia mulai bekerja didapatkan: 22.0 tahun.
Fungsi fit_median_imputation: proses fitting telah selesai, berikut hasilnya {'median_starting_age': np.float64(22.0), 'person_age': np.float64(26.0), 'person_income': np.float64(55000.0), 'loan_amnt': np.float64(8000.0), 'loan_int_rate': np.float64(10.99), 'loan_percent_income': np.float64(0.15), 'cb_person_cred_hist_length': np.float64(4.0)}.


In [28]:
serialize_data(imputer_median_dict, config["path_imputer_median"])

In [29]:
X_train = transform_median_imputation(X_train, imputer_median_dict) 

Fungsi transform_median_imputation: parameter telah divalidasi.
Fungsi transform_median_imputation: informasi count na sebelum dilakukan imputasi:
person_age                       3
person_income                    0
loan_amnt                        0
loan_int_rate                 2474
loan_percent_income              0
cb_person_cred_hist_length       0
person_emp_length              708
dtype: int64

Fungsi transform_median_imputation: informasi count na setelah dilakukan imputasi:
person_age                    0
person_income                 0
loan_amnt                     0
loan_int_rate                 0
loan_percent_income           0
cb_person_cred_hist_length    0
person_emp_length             0
dtype: int64



In [30]:
X_valid = transform_median_imputation(X_valid, imputer_median_dict)

Fungsi transform_median_imputation: parameter telah divalidasi.
Fungsi transform_median_imputation: informasi count na sebelum dilakukan imputasi:
person_age                      0
person_income                   0
loan_amnt                       0
loan_int_rate                 319
loan_percent_income             0
cb_person_cred_hist_length      0
person_emp_length              94
dtype: int64

Fungsi transform_median_imputation: informasi count na setelah dilakukan imputasi:
person_age                    0
person_income                 0
loan_amnt                     0
loan_int_rate                 0
loan_percent_income           0
cb_person_cred_hist_length    0
person_emp_length             0
dtype: int64



In [31]:
X_test = transform_median_imputation(X_test, imputer_median_dict)

Fungsi transform_median_imputation: parameter telah divalidasi.
Fungsi transform_median_imputation: informasi count na sebelum dilakukan imputasi:
person_age                      2
person_income                   0
loan_amnt                       0
loan_int_rate                 312
loan_percent_income             0
cb_person_cred_hist_length      0
person_emp_length              90
dtype: int64

Fungsi transform_median_imputation: informasi count na setelah dilakukan imputasi:
person_age                    0
person_income                 0
loan_amnt                     0
loan_int_rate                 0
loan_percent_income           0
cb_person_cred_hist_length    0
person_emp_length             0
dtype: int64



In [32]:
X_train.isna().sum()

person_age                    0
person_income                 0
person_home_ownership         0
person_emp_length             0
loan_intent                   0
loan_grade                    0
loan_amnt                     0
loan_int_rate                 0
loan_percent_income           0
cb_person_default_on_file     0
cb_person_cred_hist_length    0
dtype: int64

In [33]:
X_valid.isna().sum()

person_age                    0
person_income                 0
person_home_ownership         0
person_emp_length             0
loan_intent                   0
loan_grade                    0
loan_amnt                     0
loan_int_rate                 0
loan_percent_income           0
cb_person_default_on_file     0
cb_person_cred_hist_length    0
dtype: int64

In [34]:
X_test.isna().sum()

person_age                    0
person_income                 0
person_home_ownership         0
person_emp_length             0
loan_intent                   0
loan_grade                    0
loan_amnt                     0
loan_int_rate                 0
loan_percent_income           0
cb_person_default_on_file     0
cb_person_cred_hist_length    0
dtype: int64

In [35]:
def float_convert(data: pd.DataFrame, num_cols: List[str]) -> pd.DataFrame:

    if not isinstance(data, pd.DataFrame):
        raise TypeError("Fungsi float_convert: parameter data harus bertipe DataFrame.")
    if not isinstance(num_cols, list):
        raise TypeError("Fungsi float_convert: parameter num_cols harus bertipe list.")
    
    print("Fungsi float_convert: parameter telah divalidasi.")
    data = data.copy()

    valid_cols = [col for col in num_cols if col in data.columns]

    print("Fungsi float_convert: tipe data SEBELUM konversi:")
    print(data[valid_cols].dtypes)
    print("")

    for col in valid_cols:
        data[col] = data[col].astype("float64")

    print("Fungsi float_convert: tipe data SESUDAH konversi:")
    print(data[valid_cols].dtypes)
    print("")

    return data

In [36]:
X_train = float_convert(X_train, config["columns_num"])

Fungsi float_convert: parameter telah divalidasi.
Fungsi float_convert: tipe data SEBELUM konversi:
person_age                    float64
person_income                   int64
person_emp_length             float64
loan_amnt                       int64
loan_int_rate                 float64
loan_percent_income           float64
cb_person_cred_hist_length      int64
dtype: object

Fungsi float_convert: tipe data SESUDAH konversi:
person_age                    float64
person_income                 float64
person_emp_length             float64
loan_amnt                     float64
loan_int_rate                 float64
loan_percent_income           float64
cb_person_cred_hist_length    float64
dtype: object



In [37]:
X_valid = float_convert(X_valid, config["columns_num"])

Fungsi float_convert: parameter telah divalidasi.
Fungsi float_convert: tipe data SEBELUM konversi:
person_age                    float64
person_income                   int64
person_emp_length             float64
loan_amnt                       int64
loan_int_rate                 float64
loan_percent_income           float64
cb_person_cred_hist_length      int64
dtype: object

Fungsi float_convert: tipe data SESUDAH konversi:
person_age                    float64
person_income                 float64
person_emp_length             float64
loan_amnt                     float64
loan_int_rate                 float64
loan_percent_income           float64
cb_person_cred_hist_length    float64
dtype: object



In [38]:
X_test = float_convert(X_test, config["columns_num"])

Fungsi float_convert: parameter telah divalidasi.
Fungsi float_convert: tipe data SEBELUM konversi:
person_age                    float64
person_income                   int64
person_emp_length             float64
loan_amnt                       int64
loan_int_rate                 float64
loan_percent_income           float64
cb_person_cred_hist_length      int64
dtype: object

Fungsi float_convert: tipe data SESUDAH konversi:
person_age                    float64
person_income                 float64
person_emp_length             float64
loan_amnt                     float64
loan_int_rate                 float64
loan_percent_income           float64
cb_person_cred_hist_length    float64
dtype: object



In [39]:
save_to_config(
    key="path_train_clean",
    value=["data/processed/X_train_clean.pkl", "data/processed/y_train_clean.pkl"]
)

Berhasil menyimpan permanen: 'path_train_clean' ke config.yaml


In [40]:
save_to_config(
    key="path_valid_clean",
    value=["data/processed/X_valid_clean.pkl", "data/processed/y_valid_clean.pkl"]
)

Berhasil menyimpan permanen: 'path_valid_clean' ke config.yaml


In [41]:
save_to_config(
    key="path_test_clean",
    value=["data/processed/X_test_clean.pkl", "data/processed/y_test_clean.pkl"]
)

Berhasil menyimpan permanen: 'path_test_clean' ke config.yaml


In [42]:
config = load_config()

In [43]:
config

{'columns_cat': ['person_home_ownership',
  'loan_intent',
  'loan_grade',
  'cb_person_default_on_file'],
 'columns_num': ['person_age',
  'person_income',
  'person_emp_length',
  'loan_amnt',
  'loan_int_rate',
  'loan_percent_income',
  'cb_person_cred_hist_length'],
 'columns_predictors': ['person_age',
  'person_income',
  'person_emp_length',
  'loan_amnt',
  'loan_int_rate',
  'loan_percent_income',
  'cb_person_cred_hist_length',
  'person_home_ownership',
  'loan_intent',
  'loan_grade',
  'cb_person_default_on_file'],
 'path_imputer_median': 'models/imputer_median.pkl',
 'path_imputer_mode': 'models/imputer_mode.pkl',
 'path_raw_data': 'data/raw/credit_risk_dataset.csv',
 'path_test_clean': ['data/processed/X_test_clean.pkl',
  'data/processed/y_test_clean.pkl'],
 'path_test_set': ['data/interim/X_test.pkl', 'data/interim/y_test.pkl'],
 'path_train_clean': ['data/processed/X_train_clean.pkl',
  'data/processed/y_train_clean.pkl'],
 'path_train_set': ['data/interim/X_train.pk

In [44]:
def fit_mode_imputation(data: pd.DataFrame, cat_cols: List[str]) -> Dict[str, str]:

    if not isinstance(data, pd.DataFrame):
        raise TypeError("Fungsi fit_mode_imputation: parameter data harus bertipe DataFrame.")
    if not isinstance(cat_col, list):
        raise TypeError("Fungsi fit_mode_imputation: parameter cat_cols harus bertipe list.")
    
    print("Fungsi fit_mode_imputation: parameter telah divalidasi.")

    imputation_data = {}
    for col in cat_cols:
        if col in data.columns:
            imputation_data[col] = data[col].mode()[0]
    print(f"Fungsi fit_mode_imputation: proses fitting selesai, hasil: {imputation_data}")
    return imputation_data

In [45]:
def transform_mode_imputation(data: pd.DataFrame, imputation_data: Dict[str, str]) -> pd.DataFrame:

    if not isinstance(data, pd.DataFrame):
        raise TypeError("Fungsi transform_mode_imputation: parameter data harus bertipe DataFrame.")
    if not isinstance(imputation_data, dict):
        raise TypeError("Fungsi transform_mode_imputation: parameter imputation_data harus bertipe dict.")
    
    print("Fungsi transform_mode_imputation: parameter telah divalidasi.")
    data = data.copy()
    
    valid_cols = list(imputation_data.keys())

    print("Fungsi transform_mode_imputation: count na SEBELUM imputasi:")
    print(data[valid_cols].isna().sum())
    print("")

    data.fillna(imputation_data, inplace=True)
    
    print("Fungsi transform_mode_imputation: count na SESUDAH imputasi:")
    print(data[valid_cols].isna().sum())
    print("")

    return data

In [46]:
def object_convert(data: pd.DataFrame, cat_cols: List[str]) -> pd.DataFrame:

    if not isinstance(data, pd.DataFrame):
        raise TypeError("Fungsi object_convert: parameter data harus bertipe DataFrame.")
    if not isinstance(cat_cols, list):
        raise TypeError("Fungsi object_convert: parameter cat_cols harus bertipe list.")

    print("Fungsi object_convert: memulai konversi")
    data = data.copy()

    valid_cols = [col for col in cat_cols if col in data.columns]

    print("Fungsi object_convert: tipe data SEBELUM konversi:")
    print(data[valid_cols].dtypes)
    print("")
    
    for col in valid_cols:
        data[col] = data[col].astype("object")
    
    print("Fungsi cast_to_object: tipe data SESUDAH konversi:")
    print(data[valid_cols].dtypes)
    print("")
    
    print("Fungsi object_convert: konversi selesai.")
    return data

In [47]:
save_to_config("path_imputer_mode", "models/imputer_mode.pkl")

Berhasil menyimpan permanen: 'path_imputer_mode' ke config.yaml


In [48]:
config = load_config()

In [49]:
imputer_mode_dict = fit_mode_imputation(X_train, config["columns_cat"])

Fungsi fit_mode_imputation: parameter telah divalidasi.
Fungsi fit_mode_imputation: proses fitting selesai, hasil: {'person_home_ownership': 'RENT', 'loan_intent': 'EDUCATION', 'loan_grade': 'A', 'cb_person_default_on_file': 'N'}


In [50]:
serialize_data(
    imputer_mode_dict,
    config["path_imputer_mode"]
)

In [51]:
X_train = transform_mode_imputation(X_train, imputer_mode_dict)

Fungsi transform_mode_imputation: parameter telah divalidasi.
Fungsi transform_mode_imputation: count na SEBELUM imputasi:
person_home_ownership        0
loan_intent                  0
loan_grade                   0
cb_person_default_on_file    0
dtype: int64

Fungsi transform_mode_imputation: count na SESUDAH imputasi:
person_home_ownership        0
loan_intent                  0
loan_grade                   0
cb_person_default_on_file    0
dtype: int64



In [52]:
X_valid = transform_mode_imputation(X_valid, imputer_mode_dict)

Fungsi transform_mode_imputation: parameter telah divalidasi.
Fungsi transform_mode_imputation: count na SEBELUM imputasi:
person_home_ownership        0
loan_intent                  0
loan_grade                   0
cb_person_default_on_file    0
dtype: int64

Fungsi transform_mode_imputation: count na SESUDAH imputasi:
person_home_ownership        0
loan_intent                  0
loan_grade                   0
cb_person_default_on_file    0
dtype: int64



In [53]:
X_test = transform_mode_imputation(X_test, imputer_mode_dict)

Fungsi transform_mode_imputation: parameter telah divalidasi.
Fungsi transform_mode_imputation: count na SEBELUM imputasi:
person_home_ownership        0
loan_intent                  0
loan_grade                   0
cb_person_default_on_file    0
dtype: int64

Fungsi transform_mode_imputation: count na SESUDAH imputasi:
person_home_ownership        0
loan_intent                  0
loan_grade                   0
cb_person_default_on_file    0
dtype: int64



In [54]:
X_train = object_convert(X_train, config["columns_cat"])

Fungsi object_convert: memulai konversi
Fungsi object_convert: tipe data SEBELUM konversi:
person_home_ownership        str
loan_intent                  str
loan_grade                   str
cb_person_default_on_file    str
dtype: object

Fungsi cast_to_object: tipe data SESUDAH konversi:
person_home_ownership        object
loan_intent                  object
loan_grade                   object
cb_person_default_on_file    object
dtype: object

Fungsi object_convert: konversi selesai.


In [55]:
X_valid = object_convert(X_valid, config["columns_cat"])

Fungsi object_convert: memulai konversi
Fungsi object_convert: tipe data SEBELUM konversi:
person_home_ownership        str
loan_intent                  str
loan_grade                   str
cb_person_default_on_file    str
dtype: object

Fungsi cast_to_object: tipe data SESUDAH konversi:
person_home_ownership        object
loan_intent                  object
loan_grade                   object
cb_person_default_on_file    object
dtype: object

Fungsi object_convert: konversi selesai.


In [56]:
X_test = object_convert(X_test, config["columns_cat"])

Fungsi object_convert: memulai konversi
Fungsi object_convert: tipe data SEBELUM konversi:
person_home_ownership        str
loan_intent                  str
loan_grade                   str
cb_person_default_on_file    str
dtype: object

Fungsi cast_to_object: tipe data SESUDAH konversi:
person_home_ownership        object
loan_intent                  object
loan_grade                   object
cb_person_default_on_file    object
dtype: object

Fungsi object_convert: konversi selesai.


In [57]:
path_X_train_clean, path_y_train_clean = config["path_train_clean"]
path_X_valid_clean, path_y_valid_clean = config["path_valid_clean"]
path_X_test_clean, path_y_test_clean = config["path_test_clean"]

In [58]:
serialize_data(X_train, path_X_train_clean)
serialize_data(y_train, path_y_train_clean)

serialize_data(X_valid, path_X_valid_clean)
serialize_data(y_valid, path_y_valid_clean)

serialize_data(X_test, path_X_test_clean)
serialize_data(y_test, path_y_test_clean)

In [59]:
X_train.isna().sum()

person_age                    0
person_income                 0
person_home_ownership         0
person_emp_length             0
loan_intent                   0
loan_grade                    0
loan_amnt                     0
loan_int_rate                 0
loan_percent_income           0
cb_person_default_on_file     0
cb_person_cred_hist_length    0
dtype: int64

In [60]:
X_train.dtypes

person_age                    float64
person_income                 float64
person_home_ownership          object
person_emp_length             float64
loan_intent                    object
loan_grade                     object
loan_amnt                     float64
loan_int_rate                 float64
loan_percent_income           float64
cb_person_default_on_file      object
cb_person_cred_hist_length    float64
dtype: object

In [61]:
X_valid.isna().sum()

person_age                    0
person_income                 0
person_home_ownership         0
person_emp_length             0
loan_intent                   0
loan_grade                    0
loan_amnt                     0
loan_int_rate                 0
loan_percent_income           0
cb_person_default_on_file     0
cb_person_cred_hist_length    0
dtype: int64

In [62]:
X_valid.dtypes

person_age                    float64
person_income                 float64
person_home_ownership          object
person_emp_length             float64
loan_intent                    object
loan_grade                     object
loan_amnt                     float64
loan_int_rate                 float64
loan_percent_income           float64
cb_person_default_on_file      object
cb_person_cred_hist_length    float64
dtype: object

In [63]:
X_test.isna().sum()

person_age                    0
person_income                 0
person_home_ownership         0
person_emp_length             0
loan_intent                   0
loan_grade                    0
loan_amnt                     0
loan_int_rate                 0
loan_percent_income           0
cb_person_default_on_file     0
cb_person_cred_hist_length    0
dtype: int64

In [64]:
X_test.dtypes

person_age                    float64
person_income                 float64
person_home_ownership          object
person_emp_length             float64
loan_intent                    object
loan_grade                     object
loan_amnt                     float64
loan_int_rate                 float64
loan_percent_income           float64
cb_person_default_on_file      object
cb_person_cred_hist_length    float64
dtype: object